# EDI Historical Protocol Reproduction and Pairing Audit v2
Run all cells in a CPU Colab runtime. Uses recovered original comparison functions embedded as evidence; no notebook upload or protocol-registry editing is required. Reads prior v1 pair plan and existing arrays. Historical reproduction is separate from configuration-correct pairing and map/model provenance. No retraining, attribution generation or canonical integration. Upload the final compact ZIP.

The historical function includes float32 normalization, shape-dependent resize, an allclose identity shortcut and available-component EDI fallback. All these outcomes remain visible. Current-library reproduction does not prove the historical environment or map-generation lineage. All four methods are covered. Corrected scientific policy is not silently substituted for historical behavior.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, zipfile, re, time, platform, sys
import numpy as np
import pandas as pd
import scipy, skimage
from skimage.metrics import structural_similarity
from functools import lru_cache
from IPython.display import display
ROOT=Path('/content/drive/MyDrive/Dissertation')
RUN_ID='edi_historical_protocol_pairing_audit_v2'
OUT=ROOT/'provenance_inventory'/RUN_ID
TABLE=OUT/'tables'; TABLE.mkdir(parents=True,exist_ok=True)
PREV=ROOT/'provenance_inventory/edi_internal_attribution_provenance_integration_v1'
REC=ROOT/'provenance_inventory/edi_gradcam_remediation_provenance_recovery_v1'
FOLDER=ROOT/'provenance_inventory/edi_dissertation_folder_artifact_map_v1'
SOURCE=PREV/'tables/internal_manifest_provenance_corrected.csv'
INVENTORY=REC/'tables/table_map_inventory.csv'
PROTOCOL_FILE=OUT/'protocol_registry.json'
K=['experiment_id','image_id','preprocess','resolution','attribution_method']
METHODS=['eigencam','ig','scorecam','shap']
TOL=1e-4
MAX_NOTEBOOK_BYTES=20_000_000
MAX_NOTEBOOKS=100

def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def atomic_json(p,x):
 p=Path(p);t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(x,indent=2,allow_nan=False));os.replace(t,p)
def atomic_csv(p,x):
 p=Path(p);t=p.with_name(p.name+'.tmp');x.to_csv(t,index=False);os.replace(t,p)
def read(p):
 if not Path(p).is_file():raise FileNotFoundError(f'Required exact source missing: {p}')
 return pd.read_csv(p,low_memory=False)
def flag(x):return str(x).lower() in ('true','1','1.0')
def iid(x):return str(x).strip().upper().replace('-','_')
def method(x):
 x=str(x).lower().replace('_','').replace('-','')
 return {'integratedgradients':'ig','integratedgradient':'ig'}.get(x,x)
def parse_map(path):
 p=Path(path)
 mt=None
 for part in reversed(p.parts):
  q=method(part)
  if q in METHODS:mt=q;break
 if mt is None:
  low=p.name.lower().replace('_','').replace('-','')
  mt=next((m for m in ['eigencam','scorecam','shap'] if m in low),None)
  if mt is None and re.search(r'_ig_',p.name,re.I):mt='ig'
 im=re.search(r'ISIC[_-]?\d+(?:_downsampled)?',p.name,re.I)
 exp=next((x for x in reversed(p.parts) if re.search(r'_r(?:128|160|192|224)_',x)),None)
 return (exp,mt,iid(im.group()) if im else None)
A=read(SOURCE)
A['source_image_id']=A.image_id.astype(str);A.image_id=A.image_id.map(iid)
A.resolution=pd.to_numeric(A.resolution,errors='raise').astype(int)
A.attribution_method=A.attribution_method.map(method)
if len(A)!=23040 or A.duplicated(K).any():raise ValueError('Internal source row/key gate failed')

SOURCE_HASHES={str(p):sha(p) for p in [SOURCE]}
ENV={'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'skimage':skimage.__version__}
D=A[A.attribution_method.isin(METHODS)].copy()
if len(D)!=15360:raise ValueError('Remaining-method count failed')
V1=ROOT/'provenance_inventory/edi_remaining_method_protocol_pair_audit_v1'
P=read(V1/'tables/table_refreshed_pair_plan.csv')
P.image_id=P.image_id.map(iid)
if len(P)!=15360 or P.duplicated(K).any():raise ValueError('Prior plan gate failed')
SOURCE_HASHES[str(V1/'tables/table_refreshed_pair_plan.csv')]=sha(V1/'tables/table_refreshed_pair_plan.csv')


The exact three historical functions below were recovered from all four supplied generation notebooks and are identical. No attribution-generation code is executed. Stored pair paths and every content-distinct intended candidate remain separate; no candidate is selected because its metrics happen to match.

In [ ]:
from skimage.transform import resize
from skimage.metrics import structural_similarity as ssim
def normalize_map(h):
    h = np.nan_to_num(h.astype(np.float32))

    h_min = h.min()
    h_max = h.max()

    denom = h_max - h_min

    if denom > 0:
        h = (h - h_min) / denom
    else:
        h = np.zeros_like(h, dtype=np.float32)

    return h.astype(np.float32)

def safe_pearson(flat1, flat2):
    flat1 = np.asarray(flat1, dtype=np.float32)
    flat2 = np.asarray(flat2, dtype=np.float32)

    if flat1.size != flat2.size:
        return np.nan

    if np.std(flat1) == 0 or np.std(flat2) == 0:
        return np.nan

    return float(np.corrcoef(flat1, flat2)[0, 1])

def compare_heatmaps(h1, h2):
    h1 = normalize_map(h1)
    h2 = normalize_map(h2)

    if h1.shape != h2.shape:
        h2 = resize(
            h2,
            h1.shape,
            preserve_range=True,
            anti_aliasing=True
        ).astype(np.float32)

    # If both maps are effectively identical, no drift.
    if np.allclose(h1, h2, atol=1e-6):
        return {
            "pearson_corr": 1.0,
            "ssim": 1.0,
            "edi": 0.0
        }

    flat1 = h1.flatten()
    flat2 = h2.flatten()

    corr = safe_pearson(flat1, flat2)

    try:
        ssim_val = float(ssim(h1, h2, data_range=1.0))
    except Exception:
        ssim_val = np.nan

    drift_scores = []

    if not np.isnan(corr):
        # Convert corr [-1,1] to similarity [0,1], then drift [1,0]
        corr_similarity = (corr + 1.0) / 2.0
        drift_scores.append(1.0 - corr_similarity)

    if not np.isnan(ssim_val):
        drift_scores.append(1.0 - ssim_val)

    if len(drift_scores) == 0:
        edi = np.nan
    else:
        edi = float(np.mean(drift_scores))

    return {
        "pearson_corr": corr,
        "ssim": ssim_val,
        "edi": edi
    }
EVIDENCE=[{'method': 'eigencam', 'source_notebook': 'GradCamPP__Eigencam (1).ipynb', 'source_sha256': '1733647c6c9dba7ba0f5c2b4cf8ab3d5ce87e139261f81e84b05c1fe73f0c66c', 'cell_index': 9, 'comparison_source': 'def normalize_map(h):\n    h = np.nan_to_num(h.astype(np.float32))\n\n    h_min = h.min()\n    h_max = h.max()\n\n    denom = h_max - h_min\n\n    if denom > 0:\n        h = (h - h_min) / denom\n    else:\n        h = np.zeros_like(h, dtype=np.float32)\n\n    return h.astype(np.float32)\n\ndef safe_pearson(flat1, flat2):\n    flat1 = np.asarray(flat1, dtype=np.float32)\n    flat2 = np.asarray(flat2, dtype=np.float32)\n\n    if flat1.size != flat2.size:\n        return np.nan\n\n    if np.std(flat1) == 0 or np.std(flat2) == 0:\n        return np.nan\n\n    return float(np.corrcoef(flat1, flat2)[0, 1])\n\ndef compare_heatmaps(h1, h2):\n    h1 = normalize_map(h1)\n    h2 = normalize_map(h2)\n\n    if h1.shape != h2.shape:\n        h2 = resize(\n            h2,\n            h1.shape,\n            preserve_range=True,\n            anti_aliasing=True\n        ).astype(np.float32)\n\n    # If both maps are effectively identical, no drift.\n    if np.allclose(h1, h2, atol=1e-6):\n        return {\n            "pearson_corr": 1.0,\n            "ssim": 1.0,\n            "edi": 0.0\n        }\n\n    flat1 = h1.flatten()\n    flat2 = h2.flatten()\n\n    corr = safe_pearson(flat1, flat2)\n\n    try:\n        ssim_val = float(ssim(h1, h2, data_range=1.0))\n    except Exception:\n        ssim_val = np.nan\n\n    drift_scores = []\n\n    if not np.isnan(corr):\n        # Convert corr [-1,1] to similarity [0,1], then drift [1,0]\n        corr_similarity = (corr + 1.0) / 2.0\n        drift_scores.append(1.0 - corr_similarity)\n\n    if not np.isnan(ssim_val):\n        drift_scores.append(1.0 - ssim_val)\n\n    if len(drift_scores) == 0:\n        edi = np.nan\n    else:\n        edi = float(np.mean(drift_scores))\n\n    return {\n        "pearson_corr": corr,\n        "ssim": ssim_val,\n        "edi": edi\n    }', 'function_sha256': '9e5183bfb1428b124d4cdc30d6cbac20cd9a231116bf9a1fcd1e9f26bae191a8'}, {'method': 'scorecam', 'source_notebook': 'Score_CAM_only (6).ipynb', 'source_sha256': 'e0c3cb97b0601754dfe7808b11dced2310552cb56f7c81dfb409b5b24f25d6dc', 'cell_index': 9, 'comparison_source': 'def normalize_map(h):\n    h = np.nan_to_num(h.astype(np.float32))\n\n    h_min = h.min()\n    h_max = h.max()\n\n    denom = h_max - h_min\n\n    if denom > 0:\n        h = (h - h_min) / denom\n    else:\n        h = np.zeros_like(h, dtype=np.float32)\n\n    return h.astype(np.float32)\n\ndef safe_pearson(flat1, flat2):\n    flat1 = np.asarray(flat1, dtype=np.float32)\n    flat2 = np.asarray(flat2, dtype=np.float32)\n\n    if flat1.size != flat2.size:\n        return np.nan\n\n    if np.std(flat1) == 0 or np.std(flat2) == 0:\n        return np.nan\n\n    return float(np.corrcoef(flat1, flat2)[0, 1])\n\ndef compare_heatmaps(h1, h2):\n    h1 = normalize_map(h1)\n    h2 = normalize_map(h2)\n\n    if h1.shape != h2.shape:\n        h2 = resize(\n            h2,\n            h1.shape,\n            preserve_range=True,\n            anti_aliasing=True\n        ).astype(np.float32)\n\n    # If both maps are effectively identical, no drift.\n    if np.allclose(h1, h2, atol=1e-6):\n        return {\n            "pearson_corr": 1.0,\n            "ssim": 1.0,\n            "edi": 0.0\n        }\n\n    flat1 = h1.flatten()\n    flat2 = h2.flatten()\n\n    corr = safe_pearson(flat1, flat2)\n\n    try:\n        ssim_val = float(ssim(h1, h2, data_range=1.0))\n    except Exception:\n        ssim_val = np.nan\n\n    drift_scores = []\n\n    if not np.isnan(corr):\n        # Convert corr [-1,1] to similarity [0,1], then drift [1,0]\n        corr_similarity = (corr + 1.0) / 2.0\n        drift_scores.append(1.0 - corr_similarity)\n\n    if not np.isnan(ssim_val):\n        drift_scores.append(1.0 - ssim_val)\n\n    if len(drift_scores) == 0:\n        edi = np.nan\n    else:\n        edi = float(np.mean(drift_scores))\n\n    return {\n        "pearson_corr": corr,\n        "ssim": ssim_val,\n        "edi": edi\n    }', 'function_sha256': '9e5183bfb1428b124d4cdc30d6cbac20cd9a231116bf9a1fcd1e9f26bae191a8'}, {'method': 'shap', 'source_notebook': 'SHAP_only (2).ipynb', 'source_sha256': '4aebe8b9b414849950ad470fb40cf57390cb32be6b02354bfb1df3f6a69a3bfc', 'cell_index': 9, 'comparison_source': 'def normalize_map(h):\n    h = np.nan_to_num(h.astype(np.float32))\n\n    h_min = h.min()\n    h_max = h.max()\n\n    denom = h_max - h_min\n\n    if denom > 0:\n        h = (h - h_min) / denom\n    else:\n        h = np.zeros_like(h, dtype=np.float32)\n\n    return h.astype(np.float32)\n\ndef safe_pearson(flat1, flat2):\n    flat1 = np.asarray(flat1, dtype=np.float32)\n    flat2 = np.asarray(flat2, dtype=np.float32)\n\n    if flat1.size != flat2.size:\n        return np.nan\n\n    if np.std(flat1) == 0 or np.std(flat2) == 0:\n        return np.nan\n\n    return float(np.corrcoef(flat1, flat2)[0, 1])\n\ndef compare_heatmaps(h1, h2):\n    h1 = normalize_map(h1)\n    h2 = normalize_map(h2)\n\n    if h1.shape != h2.shape:\n        h2 = resize(\n            h2,\n            h1.shape,\n            preserve_range=True,\n            anti_aliasing=True\n        ).astype(np.float32)\n\n    # If both maps are effectively identical, no drift.\n    if np.allclose(h1, h2, atol=1e-6):\n        return {\n            "pearson_corr": 1.0,\n            "ssim": 1.0,\n            "edi": 0.0\n        }\n\n    flat1 = h1.flatten()\n    flat2 = h2.flatten()\n\n    corr = safe_pearson(flat1, flat2)\n\n    try:\n        ssim_val = float(ssim(h1, h2, data_range=1.0))\n    except Exception:\n        ssim_val = np.nan\n\n    drift_scores = []\n\n    if not np.isnan(corr):\n        # Convert corr [-1,1] to similarity [0,1], then drift [1,0]\n        corr_similarity = (corr + 1.0) / 2.0\n        drift_scores.append(1.0 - corr_similarity)\n\n    if not np.isnan(ssim_val):\n        drift_scores.append(1.0 - ssim_val)\n\n    if len(drift_scores) == 0:\n        edi = np.nan\n    else:\n        edi = float(np.mean(drift_scores))\n\n    return {\n        "pearson_corr": corr,\n        "ssim": ssim_val,\n        "edi": edi\n    }', 'function_sha256': '9e5183bfb1428b124d4cdc30d6cbac20cd9a231116bf9a1fcd1e9f26bae191a8'}, {'method': 'ig', 'source_notebook': 'IG_only (6).ipynb', 'source_sha256': '36523ed67e59a83ffd2497230af52496a25ff54a2ddd4dff2ac01ee773555466', 'cell_index': 10, 'comparison_source': 'def normalize_map(h):\n    h = np.nan_to_num(h.astype(np.float32))\n\n    h_min = h.min()\n    h_max = h.max()\n\n    denom = h_max - h_min\n\n    if denom > 0:\n        h = (h - h_min) / denom\n    else:\n        h = np.zeros_like(h, dtype=np.float32)\n\n    return h.astype(np.float32)\n\ndef safe_pearson(flat1, flat2):\n    flat1 = np.asarray(flat1, dtype=np.float32)\n    flat2 = np.asarray(flat2, dtype=np.float32)\n\n    if flat1.size != flat2.size:\n        return np.nan\n\n    if np.std(flat1) == 0 or np.std(flat2) == 0:\n        return np.nan\n\n    return float(np.corrcoef(flat1, flat2)[0, 1])\n\ndef compare_heatmaps(h1, h2):\n    h1 = normalize_map(h1)\n    h2 = normalize_map(h2)\n\n    if h1.shape != h2.shape:\n        h2 = resize(\n            h2,\n            h1.shape,\n            preserve_range=True,\n            anti_aliasing=True\n        ).astype(np.float32)\n\n    # If both maps are effectively identical, no drift.\n    if np.allclose(h1, h2, atol=1e-6):\n        return {\n            "pearson_corr": 1.0,\n            "ssim": 1.0,\n            "edi": 0.0\n        }\n\n    flat1 = h1.flatten()\n    flat2 = h2.flatten()\n\n    corr = safe_pearson(flat1, flat2)\n\n    try:\n        ssim_val = float(ssim(h1, h2, data_range=1.0))\n    except Exception:\n        ssim_val = np.nan\n\n    drift_scores = []\n\n    if not np.isnan(corr):\n        # Convert corr [-1,1] to similarity [0,1], then drift [1,0]\n        corr_similarity = (corr + 1.0) / 2.0\n        drift_scores.append(1.0 - corr_similarity)\n\n    if not np.isnan(ssim_val):\n        drift_scores.append(1.0 - ssim_val)\n\n    if len(drift_scores) == 0:\n        edi = np.nan\n    else:\n        edi = float(np.mean(drift_scores))\n\n    return {\n        "pearson_corr": corr,\n        "ssim": ssim_val,\n        "edi": edi\n    }', 'function_sha256': '9e5183bfb1428b124d4cdc30d6cbac20cd9a231116bf9a1fcd1e9f26bae191a8'}]
atomic_json(OUT/'recovered_protocol_evidence.json',EVIDENCE)
fingerprints={}
@lru_cache(maxsize=128)
def array(path):
 x=np.load(path,allow_pickle=False)
 if not isinstance(x,np.ndarray) or x.ndim!=2:raise ValueError('Expected 2D numeric array')
 return x
def inspect(path):
 if path not in fingerprints:
  try:
   p=Path(path);x=array(path)
   fingerprints[path]={'path':path,'sha256':sha(p),'shape':json.dumps(list(x.shape)),'finite':bool(np.isfinite(x).all()),'constant':bool(np.ptp(x)==0),'error':''}
  except Exception as ex:fingerprints[path]={'path':path,'error':str(ex)}
 return fingerprints[path]
def select(value):
 paths=json.loads(value)
 if not paths:return None,'missing_from_inventory'
 info=[inspect(p) for p in paths]
 if any(x.get('error') for x in info):return None,'unreadable_candidate'
 if len({x['sha256'] for x in info})!=1:return None,'ambiguous_content'
 return paths[0],'unique_or_byte_identical'
def calculate(bp,cp):
 a=array(bp);b=array(cp)
 finite=bool(np.isfinite(a).all() and np.isfinite(b).all())
 m=compare_heatmaps(a,b)
 aa=normalize_map(a);bb=normalize_map(b)
 if aa.shape!=bb.shape:bb=resize(bb,aa.shape,preserve_range=True,anti_aliasing=True).astype(np.float32)
 shortcut=bool(np.allclose(aa,bb,atol=1e-6))
 constant=bool(np.ptp(aa)==0 or np.ptp(bb)==0)
 # Strict two-component result is separate, never substitutes missing SSIM.
 two=bool(np.isfinite(m['pearson_corr']) and np.isfinite(m['ssim']) and not constant and finite)
 primary=((1-(m['pearson_corr']+1)/2)+(1-m['ssim']))/2 if two else np.nan
 return {**m,'strict_primary_edi':primary,'identity_shortcut':shortcut,'constant_pair':constant,'nonfinite_source':not finite,'two_component_eligible':two,'ssim_unavailable':not np.isfinite(m['ssim'])}


In [ ]:
signature=hashlib.sha256(json.dumps({'sources':SOURCE_HASHES,'protocol_evidence':EVIDENCE,'environment':ENV,'tol':TOL},sort_keys=True).encode()).hexdigest()
config=OUT/'run_config.json'
if config.is_file() and json.loads(config.read_text())['signature']!=signature:raise ValueError('Changed inputs: choose new RUN_ID')
atomic_json(config,{'signature':signature,'sources':SOURCE_HASHES,'environment':ENV,'tolerance':TOL,'protocol_source':'embedded recovered functions','scope':'Historical comparison reproduction, not attribution-generation validation'})
cp=TABLE/'table_historical_pair_audit.csv'
rows=read(cp).to_dict('records') if cp.is_file() else []
done={x['row_id'] for x in rows}
src={tuple(x[k] for k in K):x for x in D.to_dict('records')}
for row in P.to_dict('records'):
 rid=hashlib.sha256(json.dumps([str(row[k]) for k in K]).encode()).hexdigest()
 if rid in done:continue
 s=src[tuple(row[k] for k in K)]
 out={**{k:row[k] for k in K},'row_id':rid,'configuration':row['configuration'],'baseline_id_mismatch':bool(row['baseline_id_mismatch']),'historical_numeric_match':False,'intended_calculated':False,'model_map_provenance_verified':False}
 rb=str(s.get('baseline_heatmap_npy',''));rc=str(s.get('comparison_heatmap_npy',''))
 out.update(recorded_baseline_path=rb,recorded_comparison_path=rc)
 try:
  inspect(rb);inspect(rc);m=calculate(rb,rc)
  out.update({'historical_'+k:v for k,v in m.items()})
  for metric,stored in [('pearson_corr','pearson_corr'),('ssim','ssim'),('edi','edi')]:
   out[metric+'_error']=abs(float(m[metric])-float(s[stored]))
  out['historical_numeric_match']=all(np.isfinite(out[k+'_error']) and out[k+'_error']<=TOL for k in ['pearson_corr','ssim','edi'])
  out['stored_primary_error']=abs(float(m['strict_primary_edi'])-float(s['edi_primary']))
 except Exception as ex:out['historical_error']=str(ex)
 bp,bs=select(row['baseline_candidates']);xp,xs=select(row['comparison_candidates'])
 out.update(intended_baseline_path=bp or '',intended_comparison_path=xp or '',baseline_selection=bs,comparison_selection=xs)
 if bp and xp:
  try:
   m=calculate(bp,xp);out.update({'intended_'+k:v for k,v in m.items()});out['intended_calculated']=True
  except Exception as ex:out['intended_error']=str(ex)
 rows.append(out);done.add(rid)
 if len(rows)%100==0:
  atomic_csv(cp,pd.DataFrame(rows));atomic_csv(TABLE/'table_map_fingerprints.csv',pd.DataFrame(fingerprints.values()))
  print(f'{len(rows):,}/15,360 comparisons',flush=True)
R=pd.DataFrame(rows)
atomic_csv(cp,R);atomic_csv(TABLE/'table_map_fingerprints.csv',pd.DataFrame(fingerprints.values()))
if len(R)!=15360 or R.row_id.duplicated().any():raise ValueError('Completion gate failed')


In [ ]:
summary=R.groupby('attribution_method').agg(comparisons=('row_id','size'),historical_numeric_matches=('historical_numeric_match',lambda x:sum(map(flag,x))),intended_calculated=('intended_calculated',lambda x:sum(map(flag,x))))
atomic_csv(TABLE/'table_method_summary.csv',summary.reset_index());display(summary)
unchanged=all(sha(p)==h for p,h in SOURCE_HASHES.items())
status={'completed_utc':datetime.now(timezone.utc).isoformat(),'audit_execution_complete':True,'comparisons_inspected':len(R),'source_files_unchanged':unchanged,'historical_comparison_code_recovered':True,'attribution_generation_independently_verified':False,'canonical_integration_authorized':False,'full_corpus_integration_authorized':False,'maps_generated':False,'models_retrained':False,'scope':'Historical function reproduction and available intended pairs only'}
atomic_json(OUT/'final_status.json',status)
archive=OUT.parent/(RUN_ID+'_compact_bundle.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(OUT.rglob('*')):
  if p.is_file() and not p.name.endswith('.tmp'):z.write(p,p.relative_to(OUT))
display(pd.Series(status));print('Upload this ZIP:',archive)
if not unchanged:raise ValueError('Source preservation failed')


Numerical agreement with a historical comparison does not resolve a configuration mismatch. Undefined SSIM, constant maps, identity shortcuts and one-component fallback are explicit historical outcomes, not accepted replacements for primary EDI. Map/model generation provenance and missing intended baselines remain subsequent recovery work. The verified CAM subset is untouched.